# Stage-3 Router — training (Colab CPU)

Trains the question-conditioned router that decides **per question** whether to use the
gaze **fovea** (DETAIL) or just the **global** thumbnail (GIST). Feasibility pilot on the
300-sample, 5-condition WearVQA run. **CPU is enough** — the router is a tiny MLP over
cached question embeddings.

Pipeline: build labels -> encode questions -> stratified+grouped CV folds -> train
(balanced sampler) -> evaluate (router P/R/F1 + accuracy-vs-tokens Pareto vs oracle /
type-prior / fixed strategies).

## 1. Install dependencies

In [ ]:
!pip install -q sentence-transformers scikit-learn matplotlib

## 2. Clone repo & enter the router folder

In [ ]:
import os
REPO = 'https://github.com/shubhamOjha1000/AAAI_2027_code.git'
if not os.path.isdir('/content/AAAI_2027_code'):
    !git clone "$REPO" /content/AAAI_2027_code
else:
    !cd /content/AAAI_2027_code && git pull --ff-only
%cd /content/AAAI_2027_code/stage3_router
!ls -la

## 3. Build labels  (DETAIL iff global wrong & fovea right)

In [ ]:
!python build_router_labels.py

## 4. Encode questions  (sentence-transformers, CPU, run once)

In [ ]:
!python extract_question_features.py

## 5. Stratified + grouped 5-fold splits

In [ ]:
!python splits.py

## 6. Train the router  (CV x seeds, class-balanced sampler, OOF predictions)

In [ ]:
!python train_router.py

## 7. Evaluate  (router metrics + accuracy-vs-tokens Pareto)

In [ ]:
!python eval_router.py

## 8. Show results

In [ ]:
import json
m = json.load(open('outputs/metrics.json'))
print('ROUTER (DETAIL class):', json.dumps(m['router'], indent=2))
print()
print('END-TO-END  [accuracy%, mean_tokens]:')
for k, v in m['end_to_end'].items():
    print(f'  {k:18s} {v}')
from IPython.display import Image, display
display(Image('outputs/pareto.png'))